# Blank Analyte Detection

**Datamine CCLAS QC Anomaly Detection**

This notebook will analyse Blank Analytes for point-blank and drift anomalies.

---

## Detection Scope

**Included:** BLK Records (`ANALYTICAL TYPE = Blank`, `QC_Type = BLK`)

**Excluded:** 
- `PRECISION_STATUS` : Always NaN
- `INTERNAL_MAX_WARNING_VALUE`, `INTERNAL_MIN_WARNING_VALUE`, `INTERNAL_MAX_WARNING_INCLUSIVE`, and `INTERNAL_MIN_WARNING_INCLUSIVE` : Not used.
- `PARENT_NUMERIC_FINAL_VALUE` : Always NaN

---

## 1. Setup & Configuration

In [1]:
%pip install scikit-learn

from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import statistics
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from IPython.display import Image, display
from io import StringIO
from sklearn.tree import export_graphviz
import pydot

In [2]:
# ----
# Configuration
# ----

CONFIG = {
    "GROUP_COLS": ["STD_LOT_CODE", "SCHEME_CODE", "ANALYTE_CODE"],
    "IGNORED_STATUSES": {"IgnoredUpperFailure", "IgnoredLowerFailure"}
}

config_table = pd.DataFrame(CONFIG.items(), columns=["Setting", "Value"])
display(config_table)

,Setting,Value
0,GROUP_COLS,"[STD_LOT_CODE, SCHEME_CODE, ANALYTE_CODE]"
1,IGNORED_STATUSES,"{IgnoredUpperFailure, IgnoredLowerFailure}"


## 2. Loading Blank Data

In [3]:
DATA_PATH = Path("../data/raw/QCAnomalyTrainingDatav3.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Training data not found: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df_raw)} Blank rows, {len(df_raw.columns)} columns.")
print(f"ANALYTICAL_TYPE: {df_raw['ANALYTICAL_TYPE'].value_counts().to_dict()}")
print(f"QC_TYPE:         {df_raw['QC_TYPE'].value_counts().to_dict()}")
print(f"STANDARD_STATUS: {df_raw['STANDARD_STATUS'].value_counts().to_dict()}")
print(f"STD_LOT_CODE:    {df_raw['STD_LOT_CODE'].value_counts().to_dict()}")
print(f"Unique analytes: {df_raw['ANALYTE_CODE'].nunique()}")

Loaded 17997 Blank rows, 28 columns.
ANALYTICAL_TYPE: {'Blank': 17997}
QC_TYPE:         {'BLK': 17997}
STANDARD_STATUS: {'Pass': 17479, 'IgnoredUpperFailure': 307, 'UpperFailure': 121, 'LowerFailure': 51, 'IgnoredLowerFailure': 35}
STD_LOT_CODE:    {'TSV_BLANK': 17997}
Unique analytes: 58


## 3. Data Preparation

In [4]:
df = df_raw[df_raw['ANALYTICAL_TYPE'] == 'Blank']

filtered_df = df.drop(['PRECISION_STATUS', 'INTERNAL_MAX_WARNING_VALUE', 'INTERNAL_MIN_WARNING_VALUE', 'INTERNAL_MAX_WARNING_INCLUSIVE', 'INTERNAL_MIN_WARNING_INCLUSIVE', 'PARENT_NUMERIC_FINAL_VALUE'], axis=1, inplace=False)

filtered_df = filtered_df.dropna(subset=["ANALYTE_CODE"])

filtered_df["ANALYSED_DATE"] = pd.to_datetime(filtered_df["ANALYSED_DATE"], dayfirst=True)

filtered_df = filtered_df.sort_values(CONFIG["GROUP_COLS"] + ["ANALYSED_DATE"]).reset_index(drop=True)

filtered_df["IS_IGNORED"] = filtered_df["STANDARD_STATUS"].isin(CONFIG["IGNORED_STATUSES"])

print(f"Blank records prepared: {len(df)} rows")
print(f"Ignored records:     {filtered_df['IS_IGNORED'].sum()} ({filtered_df['IS_IGNORED'].mean()*100:.1f}%)")


Blank records prepared: 17997 rows
Ignored records:     333 (1.9%)


## 4. Features

In [5]:
GROUP_COLUMNS = CONFIG["GROUP_COLS"]


# Target Deviation
filtered_df["TARGET_DEV"] = filtered_df["NUMERIC_FINAL_VALUE"] - filtered_df["INTERNAL_TARGET_VALUE"]
# Group Median
filtered_df["GROUP_MEDIAN"] = (
    filtered_df.groupby(GROUP_COLUMNS)["NUMERIC_FINAL_VALUE"].transform("median")
)
# ABS Median Deviation
filtered_df["ABS_MEDIAN_DEVIATION"] = (
    filtered_df["NUMERIC_FINAL_VALUE"]
    - filtered_df["GROUP_MEDIAN"]
).abs()
# Group Mad
filtered_df["GROUP_MAD"] = (
    filtered_df.groupby(GROUP_COLUMNS)["ABS_MEDIAN_DEVIATION"].transform("median")
)
# Rolling Median
filtered_df["ROLL_MEDIAN"] = (
    filtered_df.groupby(GROUP_COLUMNS, dropna=True)["NUMERIC_FINAL_VALUE"]
    .transform(lambda x: x.rolling(window=10, min_periods=3).median())
)
# Rolling Change
filtered_df["ROLLING_CHANGE"] = (
    filtered_df["ROLL_MEDIAN"]
    - filtered_df["GROUP_MEDIAN"]
)
# Z Score
filtered_df["ROB_Z_SCORE"] = np.where(
    filtered_df["GROUP_MAD"] > 0,
    0.6745 * (filtered_df["NUMERIC_FINAL_VALUE"] - filtered_df["GROUP_MEDIAN"])
    / filtered_df["GROUP_MAD"],
    np.nan,
)


## 5. Point Anomalies

In [6]:
# Test to see if NUMERIC_FINAL_VALUE IS within the threshold or not.

# There will be 2 tests. One to see if the NUMERICAL_FINAL_VALUE is within the boundries, and the other checks the Z_Score.
# If either are shown to be true, then the analyte is classified as an anomaly.

# Threshhold Test:
filtered_df["OUTSIDE_THRESHHOLD"] = (
    filtered_df["NUMERIC_FINAL_VALUE"] > filtered_df["INTERNAL_MAX_VALUE"]) | (filtered_df["NUMERIC_FINAL_VALUE"] < filtered_df["INTERNAL_MIN_VALUE"])

Threshhold_true = filtered_df["OUTSIDE_THRESHHOLD"] == True

# Z Score Test:

filtered_df["P_ZTEST"] = (
    filtered_df["ROB_Z_SCORE"] > 3
)

ZSCORE_true = filtered_df["P_ZTEST"] == True


filtered_df["P_ANOM"] = filtered_df["OUTSIDE_THRESHHOLD"] | filtered_df["P_ZTEST"]

P_true = filtered_df["P_ANOM"] == True

print(
    "Threshhold candidates:",
    Threshhold_true.sum()
)

print(
    "Z_Score candidates:",
    ZSCORE_true.sum()
)
print("-=-=-")
print(
    "Point-anomaly candidates:",
    P_true.sum()
)

display(filtered_df[P_true])

Threshhold candidates: 498
Z_Score candidates: 1151
-=-=-
Point-anomaly candidates: 1275


,ANALYTICAL_TYPE,QC_TYPE,STD_LOT_CODE,STD_CODE,JOB_CODE,NUMERIC_FINAL_VALUE,ANALYSED_DATE,SCHEME_CODE,ANALYTE_CODE,STANDARD_STATUS,...,TARGET_DEV,GROUP_MEDIAN,ABS_MEDIAN_DEVIATION,GROUP_MAD,ROLL_MEDIAN,ROLLING_CHANGE,ROB_Z_SCORE,OUTSIDE_THRESHHOLD,P_ZTEST,P_ANOM
92,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002944005,0.43000,2020-08-02 02:49:07,GE_ICP40Q12,AG,Pass,...,0.43000,0.0300,0.40000,0.0800,-0.003333,-0.033333,3.372500,False,True,True
102,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0000898337,1.01000,2020-08-08 07:15:07,GE_ICP40Q12,AG,Pass,...,1.01000,0.0300,0.98000,0.0800,-0.015000,-0.045000,8.262625,False,True,True
120,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0003759577,0.43000,2020-09-25 02:39:07,GE_ICP40Q12,AG,Pass,...,0.43000,0.0300,0.40000,0.0800,0.080000,0.050000,3.372500,False,True,True
130,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0003682043,3.61000,2020-09-28 14:42:26,GE_ICP40Q12,AG,IgnoredUpperFailure,...,3.61000,0.0300,3.58000,0.0800,0.070000,0.040000,30.183875,True,True,True
138,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0003774026,2.65000,2020-10-01 11:02:09,GE_ICP40Q12,AG,IgnoredUpperFailure,...,2.65000,0.0300,2.62000,0.0800,-0.020000,-0.050000,22.089875,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17789,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0004352211,0.46100,2020-11-05 13:26:39,GE_IMS40Q12,ZR,Pass,...,0.46100,0.0864,0.37460,0.0567,0.108500,0.022100,4.456220,False,True,True
17790,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0004352211,0.39800,2020-11-05 16:05:08,GE_IMS40Q12,ZR,Pass,...,0.39800,0.0864,0.31160,0.0567,0.144000,0.057600,3.706776,False,True,True
17816,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0005815959,0.37060,2021-01-20 15:15:24,GE_IMS40Q12,ZR,Pass,...,0.37060,0.0864,0.28420,0.0567,0.078500,-0.007900,3.380827,False,True,True
17828,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0006065977,0.43360,2021-02-12 11:21:33,GE_IMS40Q12,ZR,Pass,...,0.43360,0.0864,0.34720,0.0567,0.081600,-0.004800,4.130272,False,True,True


## 6. Drift Detection

In [11]:
filtered_df = filtered_df.sort_values("ANALYSED_DATE")

filtered_df = filtered_df.reset_index()
filtered_df = filtered_df.set_index("ANALYSED_DATE")

filtered_df["ROLL_MEDIAN_WEEK"] = (
    filtered_df.groupby(GROUP_COLUMNS, dropna=True)["NUMERIC_FINAL_VALUE"]
    .transform(lambda x: x.rolling(window="7D", min_periods=3).median())
)

filtered_df["ROLL_MEDIAN_WEEK"] = (
    filtered_df.groupby(GROUP_COLUMNS, dropna=True)["NUMERIC_FINAL_VALUE"]
    .transform(lambda x: x.rolling(window="7D", min_periods=1).median())
)


filtered_df["ROLLING_CHANGE_WEEK"] = (
    filtered_df["ROLL_MEDIAN_WEEK"]
    - filtered_df["GROUP_MEDIAN"]
)

# -=-

rolling_std = (
    filtered_df.groupby(GROUP_COLUMNS, dropna=True)["NUMERIC_FINAL_VALUE"]
    .rolling(window="7D", min_periods=1)
    .std()
    .reset_index(level=GROUP_COLUMNS, drop=True)
)
filtered_df["ROLL_STD_WEEK"] = rolling_std.values

filtered_df["ROLL_Z_SCORE_WEEK"] = (
    (filtered_df["NUMERIC_FINAL_VALUE"] - filtered_df["ROLL_MEDIAN_WEEK"]) 
    / filtered_df["ROLL_STD_WEEK"]
).abs()

filtered_df["IS_DRIFT"] = filtered_df["ROLL_Z_SCORE_WEEK"] > 10.0

drift_true = (filtered_df['IS_DRIFT'] == True) & (filtered_df["P_ANOM"] == False)
drift_false = filtered_df['IS_DRIFT'] == False

print(
    "Drift-anomaly candidates:",
    drift_true.sum()
)
print(
    "Non Drift-anomaly candidates:",
    drift_false.sum()
)



display(filtered_df)

print("-=-=-=-=-=-=-=-=-=-")

display(filtered_df[drift_true])

Drift-anomaly candidates: 2616
Non Drift-anomaly candidates: 14824


,ANALYTICAL_TYPE,QC_TYPE,STD_LOT_CODE,STD_CODE,JOB_CODE,NUMERIC_FINAL_VALUE,SCHEME_CODE,ANALYTE_CODE,STANDARD_STATUS,INTERNAL_MIN_VALUE,...,ROLLING_CHANGE,ROB_Z_SCORE,OUTSIDE_THRESHHOLD,P_ZTEST,P_ANOM,ROLL_MEDIAN_WEEK,ROLLING_CHANGE_WEEK,ROLL_STD_WEEK,ROLL_Z_SCORE_WEEK,IS_DRIFT
ANALYSED_DATE,,,,,,,,,,,,,,,,,,,,,
2020-06-26 02:38:46,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002628992,0.078000,GE_IMS40Q12,SE,Pass,-5.000,...,NaN,-0.036631,False,False,False,0.078000,-0.010509,NaN,NaN,False
2020-06-26 03:09:52,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002628992,-0.044000,GE_IMS40Q12,AS,Pass,-2.500,...,NaN,-0.619001,False,False,False,-0.044000,-0.223968,0.127279,0.000000,False
2020-06-26 03:09:52,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002628992,0.004000,GE_IMS40Q12,BI,Pass,-0.250,...,NaN,-0.505912,False,False,False,0.004000,-0.009003,0.120968,0.000000,False
2020-06-26 03:09:52,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002628992,0.001000,GE_IMS40Q12,TE,Pass,-0.500,...,NaN,-0.395781,False,False,False,0.001000,-0.007100,0.099121,0.000000,False
2020-06-26 03:09:52,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002628992,0.066000,GE_IMS40Q12,SE,Pass,-5.000,...,NaN,-0.078461,False,False,False,0.072000,-0.016509,0.116919,0.051318,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-11-22 16:34:53,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0016663011,0.011002,GE_IMS40Q12,AG,Pass,-0.125,...,-0.002497,-0.303451,False,False,False,0.011002,-0.008998,0.080996,0.000000,False
2021-11-22 16:35:09,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0016663011,-0.034207,GE_IMS40Q12,PB,Pass,-2.500,...,0.756651,-0.780038,False,False,False,-0.034207,-0.563313,0.072529,0.000000,False
2021-11-22 16:35:19,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0016663011,0.335467,GE_IMS40Q12,ZN,Pass,-12.500,...,-0.206879,-0.734143,False,False,False,0.335467,-1.286696,0.145674,0.000000,False


-=-=-=-=-=-=-=-=-=-


,ANALYTICAL_TYPE,QC_TYPE,STD_LOT_CODE,STD_CODE,JOB_CODE,NUMERIC_FINAL_VALUE,SCHEME_CODE,ANALYTE_CODE,STANDARD_STATUS,INTERNAL_MIN_VALUE,...,ROLLING_CHANGE,ROB_Z_SCORE,OUTSIDE_THRESHHOLD,P_ZTEST,P_ANOM,ROLL_MEDIAN_WEEK,ROLLING_CHANGE_WEEK,ROLL_STD_WEEK,ROLL_Z_SCORE_WEEK,IS_DRIFT
ANALYSED_DATE,,,,,,,,,,,,,,,,,,,,,
2020-06-26 12:52:53,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002610232,0.026600,GE_IMS40Q12,ZN,Pass,-12.5,...,0.299338,-0.910371,False,False,False,1.921500,0.299338,0.152922,12.391325,True
2020-06-29 09:45:34,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002670477,2.810000,GE_ICP40Q12,ZN,Pass,-12.5,...,NaN,0.439024,False,False,False,4.847500,3.137500,0.130496,15.613554,True
2020-06-29 09:45:34,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002670477,2.880000,GE_ICP40Q12,PB,Pass,-12.5,...,NaN,0.599398,False,False,False,0.800000,-0.670000,0.128490,16.188067,True
2020-06-29 09:45:34,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002670477,7.880000,GE_ICP40Q12,SB,Pass,-12.5,...,NaN,2.497519,False,False,False,3.610000,3.080000,0.126477,33.761074,True
2020-06-29 10:29:51,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0002672605,-4.860000,GE_ICP40Q12,SE,Pass,-25.0,...,NaN,-0.813652,False,False,False,-3.490000,-2.840000,0.101493,13.498472,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-09-15 12:37:56,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0013181271,1.539308,GE_IMS40Q12,CU,Pass,-5.0,...,0.203080,1.180949,False,False,False,0.923185,0.223585,0.010536,58.478509,True
2021-09-15 12:37:58,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0013181271,2.552510,GE_IMS40Q12,ZN,Pass,-12.5,...,-0.315401,0.530824,False,False,False,1.350270,-0.271892,0.006860,175.249234,True
2021-11-08 10:43:14,Blank,BLK,TSV_BLANK,TSV_BLANK,TSV_LB0015957009,-1.540308,GE_ICP40Q12,BI,Pass,-25.0,...,4.100401,-0.416153,False,False,False,1.854898,1.639898,0.175449,19.351548,True


## 7. Anomaly Detection Final

In [19]:
filtered_df["ANOMALY"] = (filtered_df['IS_DRIFT'] == True) | (filtered_df["P_ANOM"] == True)

print("Point-anomaly candidates:     ", drift_true.sum())
print("Drift candidates:             ", P_true.sum())
print("Total candidates:             ", (P_true.sum() + drift_true.sum()))
print("Non-Anomaly candidates:       ", (len(filtered_df)-(P_true.sum() + drift_true.sum())))

Point-anomaly candidates:      2616
Drift candidates:              1275
Total candidates:              3891
Non-Anomaly candidates:        13940
